In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import random

LATENT_DIM = 100
CATEGORY_DIM = 10
IMAGE_DIM = 64

# Load digits dataset
digits = load_digits()
data = digits.data
targets = digits.target

# Normalize data to [0, 1]
data = data / 16.0

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(data, targets, test_size=0.2, random_state=42)

# One-hot encode labels
onehot_encoder = OneHotEncoder(sparse_output=False, categories='auto')
y_train = onehot_encoder.fit_transform(y_train.reshape(-1, 1))
y_test = onehot_encoder.transform(y_test.reshape(-1, 1))

# Helper function to create mini-batches
def get_batches(X, y, batch_size=32):
    for i in range(0, len(X), batch_size):
        yield X[i:i + batch_size], y[i:i + batch_size]

# Generator network
class Generator(nn.Module):
    def __init__(self, hidden_units1 = 128, hidden_units2 = 256):
        super(Generator, self).__init__()
        self.fc1 = nn.Linear(LATENT_DIM + CATEGORY_DIM, hidden_units1)
        self.fc2 = nn.Linear(hidden_units1, hidden_units2)
        self.fc3 = nn.Linear(hidden_units2, IMAGE_DIM)
        self.relu = nn.ReLU()

    def forward(self, x, labels):
        x = torch.cat((x, labels), dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        # x = self.fc3(x)
        # x = (torch.tanh(self.fc3(x))+1)*0.5
        x = torch.sigmoid(self.fc3(x))
        return x

# Discriminator network
class Discriminator(nn.Module):
    def __init__(self, hidden_units1 = 256, hidden_units2 = 128):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(IMAGE_DIM, hidden_units1)
        self.fc2 = nn.Linear(hidden_units1, hidden_units2)
        self.fc3 = nn.Linear(hidden_units2, CATEGORY_DIM)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.softmax(self.fc3(x))
        return x

generator = Generator()
discriminator = Discriminator()

def display_images(data, labels=None, num_images=10):
    """
    Display the first num_images in the given dataset.

    Parameters:
    - data (numpy array): Array containing image data.
    - labels (numpy array, optional): Array containing labels.
    - num_images (int, optional): Number of images to display. Default is 10.
    """
    print("original image")
    fig, axes = plt.subplots(1, num_images, figsize=(20, 2))
    for i in range(num_images):
        image = data[i].reshape(8, 8)
        axes[i].imshow(image, cmap='gray')
        axes[i].axis('off')
        if labels is not None:
            label = np.argmax(labels[i])
            axes[i].set_title(str(label))
    plt.show()



# Evaluation
def evaluate_model(model, X, y):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    outputs = model(X)
    _, predicted = torch.max(outputs, 1)
    _, actual = torch.max(y, 1)
    accuracy = (predicted == actual).sum().item() / len(y)
    confusion_matrix = torch.zeros(10, 10)
    for t, p in zip(actual, predicted):
        confusion_matrix[t, p] += 1
    return accuracy, confusion_matrix.numpy()



In [ ]:
have_torchsummary = False
try:
    from torchsummary import summary
    have_torchsummary = True
except ModuleNotFoundError:
    have_torchsummary = False

if have_torchsummary:
    # Ensure you move the model to the appropriate device (CPU or GPU) if necessary
    generator = generator.to('cpu')
    discriminator = discriminator.to('cpu')
    
    # Visualize the Generator
    print("Generator architecture:")
    summary(generator, [(LATENT_DIM,), (CATEGORY_DIM,)])
    
    # Visualize the Discriminator
    print("Discriminator architecture:")
    summary(discriminator, (IMAGE_DIM,))
